In [2]:
# file: mass_curvature_check.py
# Purpose: compute m_lat from 2pt correlators at several betas and test m_lat^2 ~ k * mu + b

import jax
import jax.numpy as jnp
from jax import jit, vmap
from functools import partial

# Enable float64
jax.config.update("jax_enable_x64", True)

# -------------------------------
# 0) Inputs you must supply
# -------------------------------
# Example placeholders (replace with real data); T is temporal extent.
# Provide: betas (B,), correlators (B,T), curvature mu (B,)
betas = jnp.array([4.5, 5.0, 5.5, 6.0], dtype=jnp.float64)            # (B,)
mu    = jnp.array([0.112, 0.089, 0.071, 0.060], dtype=jnp.float64)    # (B,)

# Dummy toy correlators with a single-state dominance region; REPLACE with real C.
T = 64
t = jnp.arange(T, dtype=jnp.float64)
true_m = jnp.array([0.38, 0.335, 0.300, 0.275], dtype=jnp.float64)    # fake underlying masses

def toy_C(m):  # periodic cosh form: C(t)=A*(e^{-m t}+e^{-m (T-t)})
    A = 1.0
    return A*(jnp.exp(-m*t) + jnp.exp(-m*(T - t)))

C = vmap(toy_C)(true_m)  # shape (B,T). REPLACE with your measured correlators.

# If you have per-beta time windows (plateaus), set them here; else a common window.
# Format: list of (t_min, t_max) in Python ints.
plateau_windows = [(8, 20)] * betas.shape[0]  # inclusive t_min, exclusive t_max

# -------------------------------
# 1) Effective mass extractor
# -------------------------------
# Using the cosh-effective-mass definition with periodic bc:
#   cosh(m_eff(t)) = (C(t-1) + C(t+1)) / (2*C(t))

@jit
def m_eff_cosh(Ct):
    # Ct shape (T,)
    eps = 1e-12
    Cp = jnp.roll(Ct, -1)
    Cm = jnp.roll(Ct,  1)
    ratio = (Cp + Cm) / (2.0*jnp.clip(Ct, eps, jnp.inf))
    # numerical guard: clip inside [1+ε, +∞) for arccosh
    ratio = jnp.clip(ratio, 1.0 + 1e-10, 1e6)
    return jnp.arccosh(ratio)

def plateau_average(m_eff_t, tmin, tmax):
    # tmin, tmax are Python ints; m_eff_t is a JAX array
    sel = m_eff_t[tmin:tmax]
    # crude stability weights from local gradient
    # jnp.gradient works fine in eager mode
    grad = jnp.gradient(sel)
    w = 1.0 / jnp.maximum(1e-12, jnp.abs(grad))
    w = w / jnp.sum(w)
    m_hat = jnp.sum(w*sel)
    m_spread = jnp.std(sel)
    return m_hat, m_spread

def extract_m_lat(C_beta, t_window, T):
    # Pure Python index manipulation to keep slice bounds static-type-friendly
    tmin_raw, tmax_raw = t_window   # both Python ints
    # Guard away from boundaries and enforce tmin < tmax <= T//2
    tmin = max(2, min(tmin_raw, T//2 - 2))
    tmax = max(tmin + 3, min(tmax_raw, T//2))
    m_eff = m_eff_cosh(C_beta)
    m_hat, m_spread = plateau_average(m_eff, tmin, tmax)
    return m_hat, m_spread, (tmin, tmax)

# -------------------------------
# 2) Per-beta masses and errors
# -------------------------------
def extract_all_masses(C, windows, T):
    B = C.shape[0]
    m_list = []
    s_list = []
    w_list = []
    for b in range(B):
        m_hat, m_spread, used = extract_m_lat(C[b], windows[b], T)
        m_list.append(m_hat)
        s_list.append(jnp.maximum(m_spread, 1e-3))  # floor to avoid zero weights
        w_list.append(used)
    return jnp.stack(m_list), jnp.stack(s_list), w_list

m_lat, m_err, used_windows = extract_all_masses(C, plateau_windows, T)
m2 = jnp.square(m_lat)

# -------------------------------
# 3) Weighted linear fit: m^2 = k * mu + b
# -------------------------------
@jit
def fit_weighted(mu, m, sigma_m):
    y = jnp.square(m)
    var_y = jnp.square(2.0*m*jnp.maximum(sigma_m, 1e-6)) + 1e-8
    w = 1.0 / var_y
    W   = jnp.sum(w)
    Wx  = jnp.sum(w*mu)
    Wy  = jnp.sum(w*y)
    Wxx = jnp.sum(w*mu*mu)
    Wxy = jnp.sum(w*mu*y)
    denom = jnp.maximum(W*Wxx - Wx*Wx, 1e-12)
    k = (W*Wxy - Wx*Wy) / denom
    b = (Wxx*Wy - Wx*Wxy) / denom
    # uncertainties
    sigma_k2 = W / denom
    sigma_b2 = Wxx / denom
    # chi^2 and mean residual
    yfit = k*mu + b
    chi2 = jnp.sum(w * jnp.square(y - yfit))
    dof  = mu.shape[0] - 2
    chi2_red = chi2 / jnp.maximum(dof, 1)
    mean_abs_resid = jnp.mean(jnp.abs(y - yfit))
    return k, b, jnp.sqrt(sigma_k2), jnp.sqrt(sigma_b2), chi2_red, mean_abs_resid

k, b, sk, sb, chi2_red, mean_abs_resid = fit_weighted(mu, m_lat, m_err)

# -------------------------------
# 4) Report
# -------------------------------
def summarize():
    print("== Mass–Curvature Correspondence Check ==")
    print(f"B (num betas): {betas.shape[0]}, T: {T}")
    print()
    print("beta    t_min..t_max    m_lat     σ_m      mu       m_lat^2")
    for i in range(betas.shape[0]):
        tmin, tmax = used_windows[i]
        print(f"{float(betas[i]):5.2f}   {int(tmin):2d}..{int(tmax):2d}    "
              f"{float(m_lat[i]):.6f}  {float(m_err[i]):.6f}  "
              f"{float(mu[i]):.6f}  {float(m2[i]):.6f}")
    print()
    print("Weighted fit:  m_lat^2 = k * mu + b")
    print(f"k = {float(k):.6f} ± {float(sk):.6f}")
    print(f"b = {float(b):+.6e} ± {float(sb):.6e}")
    print(f"reduced χ^2 = {float(chi2_red):.3f}")
    print(f"mean |residual| = {float(mean_abs_resid):.6f}")
    ok_k = jnp.abs(k - 1.0) <= 0.05  # within 5% of 1
    ok_r = mean_abs_resid < 5e-3     # residual target
    print(f"targets: k≈1? {'OK' if bool(ok_k) else 'NO'} ;  mean|res|<0.005? {'OK' if bool(ok_r) else 'NO'}")

if __name__ == "__main__":
    summarize()


== Mass–Curvature Correspondence Check ==
B (num betas): 4, T: 64

beta    t_min..t_max    m_lat     σ_m      mu       m_lat^2
 4.50    8..20    0.380000  0.001000  0.112000  0.144400
 5.00    8..20    0.335000  0.001000  0.089000  0.112225
 5.50    8..20    0.300000  0.001000  0.071000  0.090000
 6.00    8..20    0.275000  0.001000  0.060000  0.075625

Weighted fit:  m_lat^2 = k * mu + b
k = 1.311865 ± 0.017157
b = -3.326774e-03 ± 1.384936e-03
reduced χ^2 = 2.260
mean |residual| = 0.000607
targets: k≈1? NO ;  mean|res|<0.005? OK
